# Llama-4-Scout-17B-16E-Instruct Benchmark on val_data\n
\n
This notebook runs three checks for one VLM:\n
1. Dataset suitability/usability validation\n
2. Accuracy benchmark on val_data\n
3. Raw-attention visualization smoke test\n

In [ ]:
from pathlib import Path
import json
from pprint import pprint

from finesightbench.evaluation.framework import (
    validate_val_dataset,
    evaluate_model_on_val_data,
)

MODEL_NAME = "Llama-4-Scout-17B-16E-Instruct"
VAL_ROOT = Path("/home/snt/projects_lujun/FineSightBench/data/val_data")
OUTPUT_DIR = Path("/home/snt/projects_lujun/FineSightBench/outputs/vlm_eval")
MAX_SAMPLES_PER_SPLIT = 60   # set to -1 for full split
ALLOW_DOWNLOAD = False       # gated model, requires HF access token + 217GB

SyntaxError: unexpected character after line continuation character (247886842.py, line 1)

In [ ]:
validation = validate_val_dataset(VAL_ROOT)
print(json.dumps({
    "dataset_usable": validation["dataset_usable"],
    "label_files": validation["label_files"],
    "raw_samples": validation["raw_samples"],
    "evaluable_samples": validation["evaluable_samples"],
    "missing_images": validation["missing_images"],
    "corrupted_images": validation["corrupted_images"],
}, indent=2, ensure_ascii=False))
print('\nSplit counts:')
pprint(validation['split_counts'])
print('\nTask counts:')
pprint(validation['task_counts'])

In [ ]:
report = evaluate_model_on_val_data(
    model_name=MODEL_NAME,
    val_root=VAL_ROOT,
    output_dir=OUTPUT_DIR,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
    local_files_only=not ALLOW_DOWNLOAD,
    run_attention_test=True,
)

print(json.dumps({
    "model_name": report.get("model_name"),
    "model_id": report.get("model_id"),
    "status": report.get("status"),
    "accuracy": report.get("accuracy"),
    "num_evaluated": report.get("num_evaluated"),
    "num_successful_inference": report.get("num_successful_inference"),
    "attention_test": report.get("attention_test", {}),
    "error": report.get("error", ""),
}, indent=2, ensure_ascii=False))

In [ ]:
print('accuracy_by_split:')
pprint(report.get('accuracy_by_split', {}))
print('\naccuracy_by_task:')
pprint(report.get('accuracy_by_task', {}))

if report.get('num_errors', 0) > 0:
    print(f"\nnum_errors: {report['num_errors']}")
    print('See report path:', Path(report['predictions_path']).parent / 'errors.json')

In [ ]:
from IPython.display import Image as IPImage, display

attn = report.get('attention_test', {})
if attn.get('status') == 'passed' and attn.get('gif_path'):
    print('Attention GIF:', attn['gif_path'])
    display(IPImage(filename=attn['gif_path']))
else:
    print('Attention test status:', attn.get('status'))
    print('Reason:', attn.get('reason', ''))